#### **Project:** Queens Supermarket Sales ETL Pipeline  
**Client Context:** Midroc Commerce (Queens Supermarket)  

**Objective:**  
Transform raw data from the Bronze layer into clean, enriched, and analytics-ready data in the Silver layer. 
This phase includes data cleaning, feature engineering, business metric calculations, and standardization following data engineering best practices.

**Key Deliverables:**
- Cleaned and standardized data
- Enriched features (time dimensions, flags, etc.)
- Consistent data quality
- Ready for loading into the Data Warehouse

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import json

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported successfully!")

Libraries imported successfully!


#### Load Data from Bronze Layer

In [2]:
BRONZE_PATH = '../data/bronze/queens_supermarket_sales_bronze.parquet'
SILVER_PATH = '../data/silver/'

# Create silver directory
os.makedirs(SILVER_PATH, exist_ok=True)

# Load bronze data
df_bronze = pd.read_parquet(BRONZE_PATH)

print(f"Shape: {df_bronze.shape}")
df_bronze.head()

Shape: (50000, 19)


,invoice_id,branch_id,branch_name,city,customer_type,gender,product_line,unit_price,quantity,tax_15,total,date,time,payment,cogs,gross_income,gross_margin_percentage,rating,month
0,INV100000,Q001,Queens - Bole,Addis Ababa,Member,Female,Food and Beverages,265.40,5,199.05,1526.05,2025-09-18,14:05,CBE Birr,887.41,638.64,41.85,7.40,2025-09
1,INV100001,Q001,Queens - Bole,Addis Ababa,Member,Female,Health and Beauty,221.93,10,332.90,2552.20,2025-04-21,09:58,Card,1482.98,1069.22,41.89,5.80,2025-04
2,INV100002,Q002,Queens - CMC,Addis Ababa,Member,Male,Food and Beverages,383.42,2,115.03,881.87,2025-12-14,16:49,Cash,545.90,335.97,38.10,7.60,2025-12
3,INV100003,Q002,Queens - CMC,Addis Ababa,Normal,Female,Electronic Accessories,1170.10,2,351.03,2691.23,2025-07-06,08:01,Cash,1778.77,912.46,33.90,4.60,2025-07
4,INV100004,Q001,Queens - Bole,Addis Ababa,Member,Male,Snacks and Confectionery,63.50,6,57.15,438.15,2025-05-31,19:10,Cash,279.91,158.24,36.12,4.40,2025-05


#### Data Cleaning & Standardization

In [3]:
df_silver = df_bronze.copy()

# 1. Handle missing values (if any)
print("Missing values before cleaning:", df_silver.isnull().sum().sum())

Missing values before cleaning: 0


In [4]:
# 2. Standardize text columns
df_silver['branch_name'] = df_silver['branch_name'].str.strip()
df_silver['product_line'] = df_silver['product_line'].str.strip()
df_silver['city'] = df_silver['city'].str.strip()

In [5]:
# 3. Ensure correct data types
df_silver['date'] = pd.to_datetime(df_silver['date'])
df_silver['quantity'] = df_silver['quantity'].astype('int16')

In [6]:
# 4. Create derived time features
df_silver['year'] = df_silver['date'].dt.year
df_silver['month'] = df_silver['date'].dt.month
df_silver['day'] = df_silver['date'].dt.day
df_silver['weekday'] = df_silver['date'].dt.weekday
df_silver['weekday_name'] = df_silver['date'].dt.day_name()
df_silver['is_weekend'] = df_silver['weekday'].isin([5, 6]).astype(int)

In [7]:
# 5. Create time of day category
def get_time_of_day(time_str):
    hour = int(time_str.split(':')[0])
    if 8 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df_silver['time_of_day'] = df_silver['time'].apply(get_time_of_day)

print("Data cleaning and standardization completed.")
print(f"New shape: {df_silver.shape}")

Data cleaning and standardization completed.
New shape: (50000, 25)


#### Business Metric Calculations & Feature Engineering

In [8]:
# 1. Calculate key business metrics (if not accurate)
df_silver['subtotal'] = df_silver['total'] - df_silver['tax_15']

# 2. Profitability flags
df_silver['high_value_transaction'] = (df_silver['total'] > df_silver['total'].quantile(0.95)).astype(int)
df_silver['low_margin_flag'] = (df_silver['gross_margin_percentage'] < 25).astype(int)

# 3. Customer segmentation
df_silver['customer_segment'] = pd.cut(
    df_silver['total'],
    bins=[0, 1000, 3000, 6000, float('inf')],
    labels=['Small', 'Medium', 'Large', 'VIP']
)

# 4. Branch performance category (will be useful later)
df_silver['branch_performance'] = 'Standard'
top_branches = ['Queens - Bole', 'Queens - CMC']
df_silver.loc[df_silver['branch_name'].isin(top_branches), 'branch_performance'] = 'High'

print("Business metrics and features engineered successfully.")

Business metrics and features engineered successfully.


#### Data Quality Validation



In [9]:
checks = {
    "Total Records": len(df_silver),
    "Missing Values": df_silver.isnull().sum().sum(),
    "Duplicate Invoices": df_silver.duplicated(subset=['invoice_id']).sum(),
    "Negative Totals": (df_silver['total'] < 0).sum(),
    "Invalid Quantity": (df_silver['quantity'] <= 0).sum(),
    "Invalid Gross Margin": ((df_silver['gross_margin_percentage'] < 0) | 
                            (df_silver['gross_margin_percentage'] > 100)).sum()
}

for check_name, result in checks.items():
    status = "PASS" if result == 0 else f"  FAIL ({result} issues)"
    print(f"{check_name:25}: {status}")

print("\n Data Quality Validation completed.")

Total Records            :   FAIL (50000 issues)
Missing Values           : PASS
Duplicate Invoices       : PASS
Negative Totals          : PASS
Invalid Quantity         : PASS
Invalid Gross Margin     : PASS

 Data Quality Validation completed.


#### Save to Silver Layer

In [10]:
# Save cleaned and transformed data to Silver layer
silver_file_path = os.path.join(SILVER_PATH, 'queens_supermarket_sales_silver.parquet')

df_silver.to_parquet(silver_file_path, index=False, compression='snappy')

print(f"Transformed data successfully saved to Silver layer!")
print(f"File path: {silver_file_path}")
print(f"File size: {os.path.getsize(silver_file_path) / (1024*1024):.2f} MB")

Transformed data successfully saved to Silver layer!
File path: ../data/silver/queens_supermarket_sales_silver.parquet
File size: 2.63 MB


#### Save Transformation Metadata

In [11]:
transformation_metadata = {
    "transformation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "source_layer": "bronze",
    "target_layer": "silver",
    "records_processed": len(df_silver),
    "new_features_added": [
        "year", "month", "day", "weekday", "weekday_name", "is_weekend",
        "time_of_day", "subtotal", "high_value_transaction", 
        "low_margin_flag", "customer_segment", "branch_performance"
    ],
    "transformations_applied": [
        "Data cleaning & standardization",
        "Time dimension creation",
        "Business metric calculations",
        "Customer segmentation",
        "Data quality validation"
    ]
}

metadata_path = os.path.join(SILVER_PATH, 'transformation_metadata.json')

with open(metadata_path, 'w') as f:
    json.dump(transformation_metadata, f, indent=4)

print(f"Transformation metadata saved successfully!")

Transformation metadata saved successfully!


#### Final Verification

In [13]:
df_silver_verified = pd.read_parquet(silver_file_path)

print("Silver Layer Sample Data:")
display(df_silver_verified.head(3))

print(f"\nFinal Silver Layer Shape: {df_silver_verified.shape}")
print("Data Transformation completed successfully!")

Silver Layer Sample Data:


,invoice_id,branch_id,branch_name,city,customer_type,gender,product_line,unit_price,quantity,tax_15,total,date,time,payment,cogs,gross_income,gross_margin_percentage,rating,month,year,day,weekday,weekday_name,is_weekend,time_of_day,subtotal,high_value_transaction,low_margin_flag,customer_segment,branch_performance
0,INV100000,Q001,Queens - Bole,Addis Ababa,Member,Female,Food and Beverages,265.40,5,199.05,1526.05,2025-09-18,14:05,CBE Birr,887.41,638.64,41.85,7.40,9,2025,18,3,Thursday,0,Afternoon,1327.00,0,0,Medium,High
1,INV100001,Q001,Queens - Bole,Addis Ababa,Member,Female,Health and Beauty,221.93,10,332.90,2552.20,2025-04-21,09:58,Card,1482.98,1069.22,41.89,5.80,4,2025,21,0,Monday,0,Morning,2219.30,0,0,Medium,High
2,INV100002,Q002,Queens - CMC,Addis Ababa,Member,Male,Food and Beverages,383.42,2,115.03,881.87,2025-12-14,16:49,Cash,545.90,335.97,38.10,7.60,12,2025,14,6,Sunday,1,Afternoon,766.84,0,0,Small,High



Final Silver Layer Shape: (50000, 30)
Data Transformation completed successfully!
